Ficheiros extraidos de 

https://dados.gov.pt/pt/datasets/freguesias-de-portugal/#/resources

In [2]:
# fazer pip install geopandas
import geopandas as gpd

gdf = gpd.read_file("Cont_AAD_CAOP2017.shp")

print(gdf.crs)

EPSG:3763


In [3]:
import geopandas as gpd

# Ler shapefile
gdf = gpd.read_file("Cont_AAD_CAOP2017.shp")

# Corrigir acentos
def corrigir_acentos(texto):
    if isinstance(texto, str):
        try:
            return texto.encode('latin1').decode('utf-8')
        except:
            return texto
    return texto

# aplicar correção às colunas de texto
for col in gdf.select_dtypes(include="object"):
    gdf[col] = gdf[col].apply(corrigir_acentos)

print(gdf[["Distrito", "Concelho", "Freguesia"]].head())

# calcular centróides
centroides = gdf.copy()
centroides["geometry"] = centroides.geometry.centroid

# converter para latitude/longitude
centroides = centroides.to_crs(epsg=4326)

centroides["longitude"] = centroides.geometry.x
centroides["latitude"] = centroides.geometry.y

# dataframe final
df_freguesias = centroides[["Distrito", "Concelho", "Freguesia", "latitude", "longitude"]]

print(df_freguesias.head())

# guardar
df_freguesias.to_excel("freguesias_coordenadas.xlsx", index=False)

  Distrito       Concelho                  Freguesia
0     FARO      ALBUFEIRA  Albufeira e Olhos de Água
1     FARO  VILA DO BISPO                     Sagres
2     FARO      ALBUFEIRA  Albufeira e Olhos de Água
3     FARO      ALBUFEIRA  Albufeira e Olhos de Água
4     FARO      ALBUFEIRA  Albufeira e Olhos de Água
  Distrito       Concelho                  Freguesia   latitude  longitude
0     FARO      ALBUFEIRA  Albufeira e Olhos de Água  37.073759  -8.282108
1     FARO  VILA DO BISPO                     Sagres  37.036493  -8.944813
2     FARO      ALBUFEIRA  Albufeira e Olhos de Água  37.074520  -8.307291
3     FARO      ALBUFEIRA  Albufeira e Olhos de Água  37.074610  -8.307643
4     FARO      ALBUFEIRA  Albufeira e Olhos de Água  37.074706  -8.307764


In [4]:
df_freguesias["Freguesia"].nunique()

2706

In [7]:
df_freguesias_final = (
    df_freguesias
    .groupby(["Distrito", "Concelho", "Freguesia"], as_index=False)
    .agg({
        "latitude": "mean",
        "longitude": "mean"
    })
)

print(df_freguesias_final.head())

df_freguesias_final.to_excel("freguesias_coordenadas_media.xlsx", index=False)

  Distrito            Concelho                      Freguesia   latitude  \
0   AVEIRO  ALBERGARIA-A-VELHA  Albergaria-a-Velha e Valmaior  40.692786   
1   AVEIRO  ALBERGARIA-A-VELHA                     Alquerubim  40.633489   
2   AVEIRO  ALBERGARIA-A-VELHA                         Angeja  40.689688   
3   AVEIRO  ALBERGARIA-A-VELHA                         Branca  40.745561   
4   AVEIRO  ALBERGARIA-A-VELHA             Ribeira de Fráguas  40.745328   

   longitude  
0  -8.479692  
1  -8.500896  
2  -8.573416  
3  -8.490252  
4  -8.436200  


In [8]:
duplicadas = df_freguesias[
    df_freguesias.duplicated(["Distrito", "Concelho", "Freguesia"], keep=False)
]

print(duplicadas)

              Distrito              Concelho  \
0                 FARO             ALBUFEIRA   
1                 FARO         VILA DO BISPO   
2                 FARO             ALBUFEIRA   
3                 FARO             ALBUFEIRA   
4                 FARO             ALBUFEIRA   
...                ...                   ...   
2040            AVEIRO  SANTA MARIA DA FEIRA   
2507             BRAGA                  FAFE   
2591             BRAGA                  FAFE   
3114  VIANA DO CASTELO               CAMINHA   
3121  VIANA DO CASTELO               CAMINHA   

                                              Freguesia   latitude  longitude  
0                             Albufeira e Olhos de Água  37.073759  -8.282108  
1                                                Sagres  37.036493  -8.944813  
2                             Albufeira e Olhos de Água  37.074520  -8.307291  
3                             Albufeira e Olhos de Água  37.074610  -8.307643  
4                      